# RAG vs Fine-Tuning Strategy Map

This notebook compares **RAG**, **fine-tuning-style behavior adaptation**, and **hybrid prompting** with real model calls when a provider key is configured. It does not run a real fine-tuning job; instead, it uses few-shot examples to approximate the kind of stable behavior that fine-tuning is often used to learn.


## Learning Objectives

By the end of this notebook, you should be able to:

- Explain what RAG, fine-tuning, and hybrid strategies change.
- Distinguish parametric from non-parametric knowledge.
- Build prompts that add retrieved context or style examples.
- Diagnose missing facts and missing citations in model answers.


## Three Strategies, Two Places For Knowledge

An LLM system can get information from two places: its **model weights** (parametric knowledge) and the **runtime context** it receives with each request (non-parametric knowledge). Each strategy changes one of them, or both:

| Strategy | What it changes | Where the knowledge lives | Strongest when |
|---|---|---|---|
| **RAG** | What the model sees during a request | Outside the weights, in retrieved documents | Facts are fresh, citeable, or change often |
| **Fine-tuning** | How the model behaves | Inside the weights or adapter parameters | The problem is tone, format, refusal style, or a repeated task |
| **Hybrid** | Both | Both | Answers need grounded facts and consistent behavior |

**Parametric knowledge** is hard to audit and slow to update: a base model may know what an invoice is, but it should not be trusted with this scenario's current refund policy. **Non-parametric knowledge** is easy to update and cite: the refund policy can be edited today and retrieved tomorrow. Changing *how* a model responds, rather than *what* facts it can access, is called **behavior adaptation**. In these notebooks, few-shot examples stand in for it, so we can compare strategies before training anything.

Two common mistakes:

- **Asking fine-tuning to memorize living policy:** the model can sound confident while repeating stale information.
- **Asking RAG to fix style problems:** retrieval provides facts, but it does not make answers concise, empathetic, or consistent.

The figure below frames the three options as ways to take an exam. Fine-tuning is a closed-book exam: knowledge is baked into the model at training time. RAG is an open-book exam: the model reads external documents at answer time. RAFT, the method proposed by Zhang et al., combines both by training the model to use retrieved documents well. The hybrid in this repository is lighter: it combines retrieved documents with style examples in the prompt instead of training.

<div align="center">
  <img src="assets/01-raft-closed-vs-open-book.png" alt="Closed-book, open-book, and RAFT approaches to answering a query" width="900" style="max-width: 100%; height: auto;" />
  <p><em>Closed book (knowledge in the weights), open book (documents at answer time), and RAFT (a model trained to use documents). Source: <a href="https://arxiv.org/abs/2403.10131" target="_blank" rel="noopener noreferrer">Zhang et al., 2024</a>, CC BY 4.0</em></p>
</div>


## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    call_model,
    style_messages,
)

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | Model: {CONFIG.model} | Dry run: {CONFIG.dry_run}"
)

## Three Prompting Conditions

We will ask the same user question three ways:

- **Baseline:** no retrieved evidence and no style examples.
- **RAG-only:** retrieved policy context is supplied.
- **Style-only:** response examples are supplied, but no fresh policy context is supplied.

The contrast shows why RAG and fine-tuning solve different problems.


## How To Read The Comparison

The three prompt variants are not meant to compete as polished production prompts. Each one adds one ingredient together with the instruction that ingredient needs: the baseline has only the user question, the RAG-only prompt adds policy evidence and asks the model to answer from the policy and cite it, and the style-only prompt adds examples without policy context and asks the model to match the examples. Comparing the answers makes the failure modes easier to identify: missing facts, missing citations, or inconsistent behavior.

A practical rule of thumb:

- If the model knows the tone but misses the policy → add retrieval
- If it cites correctly but sounds inconsistent → improve examples or prompts
- If both evidence and style matter → test a hybrid path before investing in a new fine-tuning run


## Build The Prompts

The next cells build the three variants for one refund question, so the prompt differences are easy to compare.


In [ ]:
request = REQUESTS[0]
question = request["question"]
policy = next(
    document for document in DOCUMENTS if document["doc_id"] == request["expected_doc"]
)
style_seed = STYLE_EXAMPLES[:2]

print(f"Question: {question}\n")
print(f"Policy [{policy['doc_id']}]: {policy['text']}\n")
print(f"Style examples: {style_seed}")

The **baseline** has no domain evidence and no style examples.


In [ ]:
baseline_messages = [
    {"role": "system", "content": "You are a concise support assistant."},
    {"role": "user", "content": question},
]

The **RAG-only** version adds one citeable policy document to the user message. Here we pick the matching policy by hand; notebook 03 builds the retriever that finds it.


In [ ]:
rag_only_messages = [
    {
        "role": "system",
        "content": "Answer only from the provided policy context. Cite source IDs in square brackets.",
    },
    {
        "role": "user",
        "content": f"Policy context:\n[{policy['doc_id']}] {policy['text']}\n\nQuestion: {question}",
    },
]

Few-shot messages make behavior examples visible in the prompt. They are not new model weights; they are context supplied at request time. The **style-only** version adds such examples but no fresh policy context, which is why it can imitate tone but may miss facts.


In [ ]:
style_only_messages = [
    {
        "role": "system",
        "content": "Match the style of the examples. Do not invent policy details.",
    },
    *style_messages(style_seed),
    {"role": "user", "content": question},
]

Collect the variants in one dictionary so the same model-call loop can compare them.


In [ ]:
prompt_variants = {
    "baseline": baseline_messages,
    "rag_only": rag_only_messages,
    "style_only": style_only_messages,
}

for name, messages in prompt_variants.items():
    print(name, "messages =", len(messages))

## Call The Model

In dry mode, the function returns deterministic outputs. With `GROQ_API_KEY` or `MISTRAL_API_KEY`, the same cell calls the live model.


In [ ]:
dry_outputs = {
    "baseline": "A refund may be possible, but I need the invoice and payment details.",
    "rag_only": "Duplicate payments can be refunded when both payments belong to the same invoice. Provide the invoice number and both payment references within 30 days. [refund-policy]",
    "style_only": "Please send the invoice number and both payment references. Finance normally replies within three business days.",
}

responses = {}
for name, messages in prompt_variants.items():
    responses[name] = call_model(messages, dry_response=dry_outputs[name])
    print(f"{name.upper()} ({responses[name].source})")
    print(responses[name].text)
    print()

## Interpret The Difference

Compare each answer with the policy printed above. The RAG-only answer can stick to those facts and cite a specific source because the prompt supplied the source text. The baseline relies on general knowledge; style-only also receives the facts embedded in its examples. In live runs they often add plausible requirements that the policy never mentions, such as payment dates, amounts, or proof of payment, even though the style-only prompt says not to invent policy details. Style examples can shape an answer and supply facts, but they do not establish that those facts match the current policy.

Look closely at the refund style example: it mentions that finance replies within three business days. A fact hidden in a style example can leak into answers without any source to cite, which is why notebook 02 examines the risk of mixing changing facts into behavior examples.


A second request shows why citations still need checking. The MFA question gets a RAG-only prompt that asks the model to cite the policy ID. Run it a few times and check whether each answer actually contains `[mfa-reset]`: **an instruction to cite is not a guarantee**. Notebook 03 turns this check into automatic citation validation.


In [ ]:
mfa_request = REQUESTS[2]
mfa_question = mfa_request["question"]
mfa_policy = next(
    document
    for document in DOCUMENTS
    if document["doc_id"] == mfa_request["expected_doc"]
)
messages = [
    {
        "role": "system",
        "content": "Answer only from policy context and cite source IDs.",
    },
    {
        "role": "user",
        "content": f"Policy context:\n[{mfa_policy['doc_id']}] {mfa_policy['text']}\n\nQuestion: {mfa_question}",
    },
]
reply = call_model(
    messages,
    dry_response="MFA can be reset only after identity verification by support. Do not send identity documents or backup codes through chat. [mfa-reset]",
)
print("Question:", mfa_question)
print()
print(f"Answer ({reply.source}): {reply.text}")
print()
citation = f"[{mfa_policy['doc_id']}]"
print(f"Cites {citation}:", citation in reply.text)

### Exercise: Build A Hybrid Prompt

The hybrid strategy combines both ingredients. Build `hybrid_messages` for the same refund question with:

- a system instruction that asks the model to answer only from the policy context, match the style of the examples, and cite source IDs,
- the style examples from `style_seed`,
- a user message with the refund policy context and the question.

Call the model, print the answer, and check whether it cites `[refund-policy]`.


In [ ]:
# @TODO: Build hybrid_messages, call the model, and check the citation.


<details>
<summary>Show Reference Solution</summary>

```python
hybrid_messages = [
    {
        "role": "system",
        "content": "Answer only from the provided policy context, match the style of the examples, and cite source IDs in square brackets.",
    },
    *style_messages(style_seed),
    {
        "role": "user",
        "content": f"Policy context:\n[{policy['doc_id']}] {policy['text']}\n\nQuestion: {question}",
    },
]
hybrid_reply = call_model(
    hybrid_messages,
    dry_response="Please send the invoice number and both payment references within 30 days. Finance replies within three business days. [refund-policy]",
)
print("Question:", question)
print()
print(f"Answer ({hybrid_reply.source}): {hybrid_reply.text}")
print()
citation = f"[{policy['doc_id']}]"
print(f"Cites {citation}:", citation in hybrid_reply.text)
```

</details>


Compare the hybrid answer with the three earlier ones. In live runs it usually keeps the policy facts and the citation, like the RAG-only answer, without inventing requirements like the baseline and style-only answers. Whether the examples also improved the tone is harder to see on a single question, so notebook 05 measures the hybrid across all requests before recommending it.


## Check Your Understanding

1. A policy changes every week, but the response style is already acceptable. Which strategy should be tried first?
2. In a RAG-only system, which part changes at request time: the model weights or the retrieved context?
3. Which prompt variant (baseline, RAG-only, or style-only) mainly tests behavior adaptation?
4. Which response should be easiest to audit?

<details>
<summary>Show Reference Answers</summary>

1. Start with RAG. The main problem is freshness and citeable policy evidence, not learned response behavior.
2. The retrieved context changes at request time. The model weights stay the same in a RAG-only system.
3. The style-only variant. It conditions behavior without retrieving current policy, but its examples also contain facts. The comparison therefore does not isolate a pure style effect.
4. The RAG-only response is easiest to audit because it asks the model to cite source IDs from the provided policy context.

</details>


## Further Reading

- [Fine-Tuning or Retrieval? Comparing Knowledge Injection in LLMs](https://arxiv.org/abs/2312.05934) - Ovadia et al., 2023
